# Lab 05 — Credit Risk — German Credit
**Classification Track** · Intermediate · ~60 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Preprocess mixed categorical/numeric credit data
2. Handle class imbalance with class_weight
3. Evaluate with ROC-AUC
4. Rank features via permutation importance

## Datasets (this folder)
- `german_credit_data.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-02-classical-machine-learning/labs/lab-05-credit-risk-german/lab-05-credit-risk-german.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0** first — it downloads `dataset.zip` with wget, unzips it, and every code cell below reads those unzipped files.
3. **Runtime → Run all** (GPU not required for Course 2).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 (`wget dataset.zip` + `unzip`) → `Runtime → Run all`.


### Setup (dataset)

Run the next cell (Cell 0) once: it downloads `dataset.zip` with wget and unzips it next to the notebook. All code below reads these unzipped files (`german_credit_data.csv`). Skips the download when the files already exist.


In [ ]:
# Cell 0 — dataset first: wget dataset.zip + unzip (run this cell first).
import os, shutil, subprocess, urllib.request, zipfile

LAB_ID = "lab-05-credit-risk-german"
DATASET_ZIP_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-02-classical-machine-learning/labs/lab-05-credit-risk-german/bundle/dataset.zip"
NEED = ["german_credit_data.csv"]  # unzipped files used by the code below

def _have_files():
    return all(os.path.exists(f) for f in NEED)

def _wget_zip(url, dest):
    # shell equivalent: !wget -q <url> -O dataset.zip
    if shutil.which("wget"):
        subprocess.run(["wget", "-q", url, "-O", dest], check=True)
    else:  # plain Python without wget: stdlib fallback
        urllib.request.urlretrieve(url, dest)

if _have_files():
    print("dataset ready:", ", ".join(NEED))
else:
    _wget_zip(DATASET_ZIP_URL, "dataset.zip")
    # shell equivalent: !unzip -o -q dataset.zip
    with zipfile.ZipFile("dataset.zip") as z:
        z.extractall(".")
    print("downloaded + unzipped dataset.zip ->", ", ".join(NEED))


## Intermediate: The Loan Officer's Scorecard

> **Scenario:** You are the data scientist at a regional bank. The loan team wants a model that flags which applicants are likely to default, so a human officer can review risky cases before approving. You have 5,000 past applications with 21 features — a mix of numbers (loan amount, age, duration) and categories (checking account status, credit history, job). Only about 1 in 3 applicants is risky, so a lazy model that says "no risk" for everyone still looks decent. Your job is to build a model the bank can actually trust.
>
> **You will learn:** how to preprocess mixed categorical/numeric data with a ColumnTransformer, why class imbalance demands `class_weight`, how ROC-AUC measures ranking quality, and how to pick a decision threshold that matches business costs.
> **Time:** ~60 minutes. **Level:** Intermediate. **Needs:** pandas + sklearn. **Env:** 🟢 Colab only.

### Mental map

| Concept | One-line idea | Where it shows up |
|---|---|---|
| ColumnTransformer | Apply different preprocessing to different columns | OneHotEncoder for categories, StandardScaler for numbers |
| Class imbalance | Risky applicants are the minority (~33%) | `class_weight="balanced"` |
| ROC-AUC | How well the model ranks risky above safe | 0.5 = coin flip, 1.0 = perfect ranking |
| Decision threshold | The score cutoff for "approve" vs "review" | Default 0.5 is rarely optimal |
| Permutation importance | Which features actually drive the model | Shuffle one column, watch AUC drop |

---

## 1. Preprocessing mixed data: one transformer to rule them all

Credit risk modelling is where machine learning meets real money. When a bank approves a loan, it is betting that the borrower will repay. Get it right, and the bank earns interest. Get it wrong, and the bank loses the principal. A good credit risk model is worth millions — and a bad one can sink a bank.

The German Credit dataset is a classic benchmark. It contains 5,000 past loan applications, each described by 21 features: checking account status, loan duration, credit history, loan purpose, loan amount, savings, employment duration, installment percentage, personal status, age, and more. The target is binary: "Risk" (bad credit) or "No Risk" (good credit). About 33% of applicants are classified as risky — a significant minority that a lazy model could ignore.

The challenge is that the features are a mix of types. Seven are numeric (loan amount, age, duration), and 13 are categorical (checking status, credit history, job type). Machine learning models need numbers, so we must encode the categories. But we cannot just convert them to arbitrary numbers — "less_0" is not "less than" 0_to_200 in any meaningful numeric sense. We need **one-hot encoding**: each category becomes its own binary column.

The clean way to handle mixed data is a **ColumnTransformer**. It applies different preprocessing to different columns: one-hot encode the categories, standard-scale the numbers, and pass everything to the model as a single matrix. This is more maintainable than manual encoding, and it ensures the same transformations are applied consistently at training and prediction time.

We also use `stratify=y` in the train/test split to preserve the 33% risk rate in both sets. With imbalanced data, an unstratified split might produce a test set with 40% risk or 25% risk by chance, making our metrics unreliable.

Let us build the preprocessing pipeline and fit a first model.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv(r"german_credit_data.csv")
df = df.drop(columns=["CustomerID"])

y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])

cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
print(f"{len(cat_cols)} categorical, {len(num_cols)} numeric features")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"train: {len(X_tr)} rows, test: {len(X_te)} rows")
print(f"risk rate — train: {y_tr.mean():.4f}, test: {y_te.mean():.4f}")

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, random_state=42))
clf.fit(X_tr, y_tr)
print("model trained OK")


**What to notice:**
1. `stratify=y` keeps the ~33% risk rate identical in train and test — essential with imbalanced data.
2. `handle_unknown="ignore"` lets the model survive a category at test time that never appeared in training.
3. The pipeline bundles preprocessing + model, so `clf.predict(X_te)` applies the same transformations automatically.

> **Pitfall:** Never fit the encoder or scaler on the full dataset before splitting — statistics from the test set would leak into training. Fitting inside a pipeline (as here) avoids this.

---

## 2. Class imbalance: don't let the majority win

With 66.6% "No Risk", a model that always predicts "No Risk" scores 66.6% accuracy and is useless. Two honest fixes: use a metric that sees through the class ratio (ROC-AUC, next section), and tell the model to care more about the minority class with `class_weight="balanced"`.

Class imbalance is one of the most common challenges in real-world machine learning. Fraud detection, disease screening, churn prediction, spam filtering — all involve rare positive cases buried in a sea of negatives. The danger is that the model learns to ignore the minority class entirely: if 97% of transactions are legitimate, a model that always says "legitimate" is 97% accurate and worthless.

The `class_weight="balanced"` parameter tells sklearn to adjust the importance of each class inversely to its frequency. Risky applicants (33%) get a weight of about 2, while safe applicants (66.6%) get a weight of about 1. This means the model is penalised twice as much for misclassifying a risky applicant as for misclassifying a safe one. The effect is to shift the decision boundary toward the minority class, increasing the true-positive rate at the cost of more false alarms.

The key insight is that **accuracy is the wrong metric** for imbalanced data. A model that always predicts the majority class has high accuracy but zero value. Instead, we should use metrics that focus on the minority class: precision, recall, F1, and ROC-AUC. ROC-AUC is particularly useful because it is threshold-free — it evaluates the model's ability to *rank* risky applicants above safe ones, regardless of where we set the cutoff.

Let us see how class weighting affects the model's performance.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix

df = pd.read_csv(r"german_credit_data.csv")
df = df.drop(columns=["CustomerID"])
y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

for weighted in [False, True]:
    cw = "balanced" if weighted else None
    clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight=cw, random_state=42))
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    y_pred = clf.predict(X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    tpr = tp / (tp + fn)
    acc = (tp + tn) / len(y_te)
    print(f"class_weight={str(cw):8s}  AUC={auc:.4f}  TPR={tpr:.4f}  accuracy={acc:.4f}")


**What to notice:**
1. `class_weight="balanced"` boosts the true-positive rate (catches more risky applicants) at the cost of more false alarms.
2. ROC-AUC barely moves — it measures *ranking*, which the weighting doesn't change much. Accuracy moves more.
3. The trade-off is the whole game: catching more risk means reviewing more safe applicants.

> **Pitfall:** Accuracy is the wrong target here. An 80%-accurate model that misses half the defaulters is worse for the bank than a 75%-accurate model that catches most of them.

---

## 3. ROC-AUC and choosing a threshold

ROC-AUC asks: "if I pick one risky and one safe applicant at random, how often does the model score the risky one higher?" It is threshold-free — it evaluates the ranking, not any single cutoff. The cutoff (threshold) is a separate business decision: lower it to catch more risk, higher it to reduce false alarms.

The ROC curve is a plot of the true-positive rate (TPR) against the false-positive rate (FPR) at every possible threshold. A perfect model has a curve that goes straight up to (0, 1) — it catches all risky applicants with zero false alarms. A random model has a diagonal line from (0, 0) to (1, 1) — it catches risky applicants at the same rate as safe ones. The area under the curve (AUC) summarises this in a single number: 1.0 is perfect, 0.5 is random.

An AUC of 0.87 means the model ranks a random risky applicant above a random safe one about 87% of the time. That is strong performance for credit risk — human loan officers typically achieve AUCs of 0.70-0.80 on the same task.

But AUC does not tell you where to set the threshold. That is a business decision. If the bank's cost of a false alarm (reviewing a safe applicant) is low compared to the cost of a missed default, set a low threshold to catch more risk. If false alarms are expensive, set a high threshold. **Youden's J statistic** (TPR − FPR) provides a data-driven starting point: it finds the threshold that maximises the difference between true positives and false positives.

Let us compute the ROC curve, find the optimal threshold, and see how the trade-off looks.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

df = pd.read_csv(r"german_credit_data.csv")
df = df.drop(columns=["CustomerID"])
y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_te, proba):.4f}")

print("\nthreshold sweep (balanced model):")
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    yp = (proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te, yp).ravel()
    print(f"  t={t}: TPR={tp/(tp+fn):.4f}  FPR={fp/(fp+tn):.4f}  precision={tp/(tp+fp):.4f}")

fpr, tpr, thr = roc_curve(y_te, proba)
best_t = thr[np.argmax(tpr - fpr)]
print(f"\nYouden's J best threshold: {best_t:.4f} (TPR={tpr[np.argmax(tpr-fpr)]:.4f}, FPR={fpr[np.argmax(tpr-fpr)]:.4f})")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc_score(y_te, proba):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="coin flip")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve — credit risk model")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("roc_curve.png", dpi=100, bbox_inches="tight")
plt.show()


**What to notice:**
1. AUC ≈ 0.87 means the model ranks a random risky applicant above a random safe one about 87% of the time.
2. Lowering the threshold from 0.7 → 0.3 raises TPR (0.60 → 0.91) but also FPR (0.09 → 0.42).
3. Youden's J (TPR − FPR) picks the threshold that best balances the two — here ≈ 0.37, well below the default 0.5.

> **Pitfall:** The "best" threshold depends on business costs. If missing a default costs 5× a false alarm, pick a lower threshold than Youden's J suggests. There is no universally correct cutoff.

---

## Exercises (do these!)

### Exercise 1 — Risk rate by checking status
Group the full dataset by `CheckingStatus` and compute the risk rate (fraction with `Risk == "Risk"`) for each group. Which group is riskiest, and which is safest?
*Expected: no_checking ≈ 0.5665 (n=1993) is riskiest; less_0 ≈ 0.0930 (n=1398) is safest; 0_to_200 ≈ 0.2178; greater_200 ≈ 0.4164.*

**Follow-up:** What is the overall risk rate across all 5,000 applicants? Check: ≈ 0.334.

<details>
<summary>Hint</summary>

`df.groupby("CheckingStatus")["Risk"].apply(lambda s: (s == "Risk").mean())` gives the per-group rate.

</details>

### Exercise 2 — ROC-AUC of the pipeline
Train the balanced logistic regression from section 3 and compute ROC-AUC on the held-out test set. Then retrain without `class_weight` and compare.
*Expected: balanced AUC ≈ 0.8664; unbalanced AUC ≈ 0.8666 (nearly identical — AUC is a ranking metric).*

**Follow-up:** At the default 0.5 threshold, what is the balanced model's TPR? Check: ≈ 0.7545.

<details>
<summary>Hint</summary>

`roc_auc_score(y_te, proba)` where `proba = clf.predict_proba(X_te)[:, 1]`.

</details>

### Exercise 3 — Top 5 features by permutation importance
Permutation importance measures how much the model's AUC drops when one feature is randomly shuffled — a model-agnostic way to rank what matters. Compute it on the test set and list the top 5 features.
*Expected: EmploymentDuration (≈0.047), OwnsProperty (≈0.039), CheckingStatus (≈0.024), LoanDuration (≈0.016), ExistingCreditsCount (≈0.013).*

**Follow-up:** Why is permutation importance computed on the *test* set, not the training set? Check: on training data the model may have memorized noise, so shuffling a useless feature shows no drop.

<details>
<summary>Hint</summary>

`permutation_importance(clf, X_te, y_te, n_repeats=10, random_state=42, scoring="roc_auc")` returns importances aligned with `X.columns`.

</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np

df = pd.read_csv(r"german_credit_data.csv")
risk = df.groupby("CheckingStatus")["Risk"].agg(total="count", risky=lambda s: (s == "Risk").sum())
risk["risk_rate"] = risk["risky"] / risk["total"]
print(risk.sort_values("risk_rate", ascending=False).round(4))
print("overall risk rate:", round((df["Risk"] == "Risk").mean(), 4))

# --- Solution 2 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix

df = pd.read_csv(r"german_credit_data.csv")
df = df.drop(columns=["CustomerID"])
y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
for weighted in [False, True]:
    cw = "balanced" if weighted else None
    clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight=cw, random_state=42))
    clf.fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    print(f"class_weight={str(cw):8s}  AUC={roc_auc_score(y_te, proba):.4f}")

# --- Solution 3 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

df = pd.read_csv(r"german_credit_data.csv")
df = df.drop(columns=["CustomerID"])
y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
pi = permutation_importance(clf, X_te, y_te, n_repeats=10, random_state=42, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
print(imp.head(5).round(4))

# --- Follow-up 1 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np

df = pd.read_csv(r"german_credit_data.csv")
overall = (df["Risk"] == "Risk").mean()
assert abs(overall - 0.334) < 0.005, overall
print("overall risk rate:", round(overall, 4))

# --- Follow-up 2 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

df = pd.read_csv(r"german_credit_data.csv")
df = df.drop(columns=["CustomerID"])
y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_te)
tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
tpr = tp / (tp + fn)
assert abs(tpr - 0.7545) < 0.01, tpr
print("balanced TPR @0.5:", round(tpr, 4))

# --- Follow-up 3 ---
import matplotlib
matplotlib.use("Agg")
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

df = pd.read_csv(r"german_credit_data.csv")
df = df.drop(columns=["CustomerID"])
y = (df["Risk"] == "Risk").astype(int)
X = df.drop(columns=["Risk"])
cat_cols = X.select_dtypes("object").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])
clf = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
clf.fit(X_tr, y_tr)
pi = permutation_importance(clf, X_te, y_te, n_repeats=10, random_state=42, scoring="roc_auc")
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)
top5 = imp.head(5)
assert "EmploymentDuration" in top5.index and "OwnsProperty" in top5.index
print("top 5:", list(top5.index))


### What to learn next
- Gradient boosting (XGBoost/LightGBM) — the usual upgrade over logistic regression for tabular credit data.
- Calibration: making predicted probabilities match real-world frequencies.
- Fairness auditing — what happens when the model treats demographic groups differently (Lab 06).

*Files in this folder: german_credit_data.csv (5000 rows: CustomerID, CheckingStatus, LoanDuration, CreditHistory, LoanPurpose, LoanAmount, ExistingSavings, EmploymentDuration, InstallmentPercent, Sex, OthersOnLoan, CurrentResidenceDuration, OwnsProperty, Age, InstallmentPlans, Housing, ExistingCreditsCount, Job, Dependents, Telephone, ForeignWorker, Risk).*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, 
or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
